In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anirudhcv/labeled-optical-coherence-tomography-oct")

print("Path to dataset files:", path)

100%|██████████| 6.70G/6.70G [00:48<00:00, 148MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2


In [ ]:
for root, dirs, files in os.walk(path):
    print(root, "->", len(files), "files")

/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2 -> 0 files
/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2/Dataset - train+val+test -> 0 files
/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2/Dataset - train+val+test/test -> 0 files
/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2/Dataset - train+val+test/test/NORMAL -> 5139 files
/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2/Dataset - train+val+test/test/CNV -> 3746 files
/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2/Dataset - train+val+test/test/DRUSEN -> 887 files
/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical-coherence-tomography-oct/versions/2/Dataset - train+val+test/test/DME -> 1161 files
/root/.cache/kagglehub/datasets/anirudhcv/labeled-optical

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tensorflow.keras.utils import image_dataset_from_directory
import os

Data Preparation

In [ ]:
train_data_dir = os.path.join(path, "Dataset - train+val+test/train")

training_set = image_dataset_from_directory(
    train_data_dir,
    labels="inferred",
    label_mode="int",
    class_names=None,
    color_mode="rgb",
    batch_size=32,
    image_size=(224, 224),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,

)

Found 76515 files belonging to 4 classes.


In [ ]:
val_data_dir = os.path.join(path, "Dataset - train+val+test/train")

validation_set = image_dataset_from_directory(
    val_data_dir,
    labels="inferred",
    label_mode="int",
    class_names=None,
    color_mode="rgb",
    batch_size=32,
    image_size=(224, 224),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False
)

Found 76515 files belonging to 4 classes.


In [ ]:
print(training_set)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


In [ ]:
INPUT_SHAPE = (224, 224, 3)

In [ ]:
mobNet = tf.keras.applications.MobileNetV3Large(
    input_shape=INPUT_SHAPE, # Define input shape for the base model
    alpha=1.0,
    minimalistic=False,
    include_top=False, # Remove the pre-trained classification head
    weights="imagenet",
    pooling=None, # We'll add global pooling explicitly after this layer
    include_preprocessing=True,
    name="MobileNetV3Large",
)

12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = tf.keras.Sequential()


In [ ]:
model.add(mobNet)
model.add(tf.keras.layers.GlobalAveragePooling2D()) # Add a pooling layer to flatten features

In [ ]:
model.add(tf.keras.layers.Dense(units = 4, activation = 'softmax'))

In [ ]:
metrics_list = ["accuracy", tf.keras.metrics.F1Score()]

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=metrics_list
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         3,844 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,000,196 (11.44 MB)

 Trainable params: 2,975,796 (11.35 MB)

 Non-trainable params: 24,400 (95.31 KB)

In [ ]:
num_classes = 4 # The model has 4 output classes

def one_hot_encode_labels(image, label):
    return image, tf.one_hot(label, num_classes)

training_set_one_hot = training_set.map(one_hot_encode_labels)
validation_set_one_hot = validation_set.map(one_hot_encode_labels)

training_history = model.fit(x = training_set_one_hot, validation_data = validation_set_one_hot, epochs =  15)

Epoch 1/15
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 529s 192ms/step - accuracy: 0.9426 - f1_score: 0.9000 - loss: 0.1693 - val_accuracy: 0.9623 - val_f1_score: 0.9361 - val_loss: 0.1070
Epoch 2/15
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 420s 176ms/step - accuracy: 0.9744 - f1_score: 0.9550 - loss: 0.0737 - val_accuracy: 0.9811 - val_f1_score: 0.9670 - val_loss: 0.0535
Epoch 3/15
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 368s 154ms/step - accuracy: 0.9847 - f1_score: 0.9730 - loss: 0.0432 - val_accuracy: 0.9885 - val_f1_score: 0.9792 - val_loss: 0.0327
Epoch 4/15
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 433s 175ms/step - accuracy: 0.9896 - f1_score: 0.9818 - loss: 0.0290 - val_accuracy: 0.9931 - val_f1_score: 0.9877 - val_loss: 0.0198
Epoch 5/15
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 370s 155ms/step - accuracy: 0.9914 - f1_score: 0.9849 - loss: 0.0227 - val_accuracy: 0.9932 - val_f1_score: 0.9877 - val_loss: 0.0203
Epoch 6/15
2392/2392 ━━━━━━━━━━━━━━━━━━━━ 417s 174ms/step - accuracy: 0.9929 - f1_score: 0.9876 - loss: 0.0186 - val_acc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model.save("./trained_eye_disease_model.keras")

In [ ]:
training_history.history

{'accuracy': [0.9425864219665527,
  0.9743710160255432,
  0.9846827387809753,
  0.9896360039710999,
  0.9914003610610962,
  0.9929164052009583,
  0.9936352372169495,
  0.9940926432609558,
  0.9950075149536133,
  0.9945762157440186,
  0.9950990080833435,
  0.9951120615005493,
  0.995334267616272,
  0.9955694675445557,
  0.9958308935165405],
 'f1_score': [array([0.9505277 , 0.9004877 , 0.77614224, 0.9728659 ], dtype=float32),
  array([0.9781884 , 0.9638928 , 0.88939744, 0.98832583], dtype=float32),
  array([0.9872046 , 0.98183614, 0.93004775, 0.9928398 ], dtype=float32),
  array([0.99131787, 0.98661894, 0.95399976, 0.9952356 ], dtype=float32),
  array([0.99297786, 0.9891518 , 0.96167463, 0.99588764], dtype=float32),
  array([0.9936845 , 0.99144036, 0.96814156, 0.99697006], dtype=float32),
  array([0.9942776 , 0.9916204 , 0.97175956, 0.99740076], dtype=float32),
  array([0.99471587, 0.99217635, 0.9743259 , 0.9974843 ], dtype=float32),
  array([0.9956502 , 0.99353164, 0.9779636 , 0.9978177

In [ ]:
#saving history
import pickle
with open('training_history.pkl', 'wb') as f:
    pickle.dump(training_history.history, f)